# 26 GRPO 的 token ratio 为什么会产生长度偏差？

## 面试回答主线

GRPO 用同一 prompt 的多条回复 reward 构造组内相对优势，省去 value critic；常见实现把同一条序列优势广播到各 token，并对 token ratio 做 PPO 风格裁剪。若损失直接对所有 token 求和，长回答会因 token 数多而获得更大总权重，即使每 token 质量相同。面试时必须交代“按 token 平均、按 sequence 平均还是按 group 平均”。实验对同一组六条 chosen/rejected 风格回复给出不同长度和 token ratio，比较 token-sum 与 sequence-normalized surrogate，并构造长回复重复计权的失败。

**核心公式：** 组优势可为 $A_i=(r_i-\mu_g)/(\sigma_g+\epsilon)$；token surrogate 为 $\frac1{T_i}\sum_t\min(r_{it}A_i,\operatorname{clip}(r_{it})A_i)$。若漏掉 $1/T_i$，长样本权重随 $T_i$ 增长。

后续依次展示同数据基线、手写核心状态/概率、结果表、真实失败与修复。数值仅用于机制验证。


## 真实案例

数据是六条脱敏客服 prompt，每条含 chosen/rejected 回答；注意力主题会将它们映射成流式键值事件。字段语义和失败模式与真实系统一致，但样本规模不能代表线上效果。


In [1]:
import math  # 导入数学函数实现概率和复杂度公式。
import warnings  # 导入警告控制模块保持输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖产生的弃用提示。
import torch  # 导入张量和自动微分基础能力。
import torch.nn as nn  # 导入模块基类以显式定义网络。
torch.manual_seed(41)  # 固定随机种子保证输出可复现。
torch.set_num_threads(1)  # 固定小实验 CPU 线程数。
samples = [  # 定义六条可读的 prompt、候选回复或流式事件。
    {'id': 'P01', 'prompt': '支付重复扣款怎么处理？', 'chosen': '核验订单后原路退款。', 'rejected': '无需核验直接忽略。'},  # 退款决策样本。
    {'id': 'P02', 'prompt': '发现陌生转账怎么办？', 'chosen': '立即冻结并核验身份。', 'rejected': '等待下个账单周期。'},  # 账户安全样本。
    {'id': 'P03', 'prompt': '收不到登录验证码？', 'chosen': '检查手机号并重发。', 'rejected': '建议注销账户。'},  # 登录支持样本。
    {'id': 'P04', 'prompt': '地址如何修改？', 'chosen': '在发货前更新地址。', 'rejected': '永久不可修改。'},  # 售后样本。
    {'id': 'P05', 'prompt': '银行卡被盗刷？', 'chosen': '冻结卡并保留证据。', 'rejected': '继续正常使用。'},  # 风险样本。
    {'id': 'P06', 'prompt': '发票抬头写错？', 'chosen': '按规则更正抬头。', 'rejected': '删除全部订单。'},  # 账单样本。
]  # 结束可读数据定义。
print('教学实验：六条脱敏客服 prompt/候选或流式状态，只解释训练与状态机制。')  # 声明实验边界。
for row in samples:  # 逐条展示 prompt/chosen/rejected。
    print(f"{row['id']} | 问题={row['prompt']} | chosen={row['chosen']} | rejected={row['rejected']}")  # 输出真实语义样本。


教学实验：六条脱敏客服 prompt/候选或流式状态，只解释训练与状态机制。
P01 | 问题=支付重复扣款怎么处理？ | chosen=核验订单后原路退款。 | rejected=无需核验直接忽略。
P02 | 问题=发现陌生转账怎么办？ | chosen=立即冻结并核验身份。 | rejected=等待下个账单周期。
P03 | 问题=收不到登录验证码？ | chosen=检查手机号并重发。 | rejected=建议注销账户。
P04 | 问题=地址如何修改？ | chosen=在发货前更新地址。 | rejected=永久不可修改。
P05 | 问题=银行卡被盗刷？ | chosen=冻结卡并保留证据。 | rejected=继续正常使用。
P06 | 问题=发票抬头写错？ | chosen=按规则更正抬头。 | rejected=删除全部订单。


## Baseline / 基线

先运行最朴素、但同样使用这些输入和同一指标的对照，避免只看一个核心算法数字。


In [2]:
group_rewards = [1.0, 0.3, 0.8, 1.0, 0.0, 0.7]  # 定义两组客服回答的验证 reward。
token_lengths = [5, 3, 7, 6, 3, 5]  # 定义六条回复的生成 token 数。
token_ratios = [[1.08] * 5, [0.95] * 3, [1.12] * 7, [1.10] * 6, [0.90] * 3, [1.05] * 5]  # 构造每条回复逐 token 的 policy ratio。
group_means = [sum(group_rewards[:3]) / 3.0] * 3 + [sum(group_rewards[3:]) / 3.0] * 3  # 计算两个 prompt 各自的组均值。
advantages = [reward - mean for reward, mean in zip(group_rewards, group_means)]  # 计算组内相对优势。
token_sum_terms = [sum(min(ratio * advantage, max(0.8, min(1.2, ratio)) * advantage) for ratio in ratios) for ratios, advantage in zip(token_ratios, advantages)]  # 错误地直接累加每个 token 的 surrogate。
baseline_metric = max(abs(value) for value in token_sum_terms)  # 记录 token-sum 下的最大样本权重。
print(f'GRPO token-sum terms={ [round(value, 3) for value in token_sum_terms] }，长度={token_lengths}，最大绝对权重={baseline_metric:.3f}')  # 展示长序列被重复加权。


GRPO token-sum terms=[1.62, -1.14, 0.784, 2.86, -1.53, 0.7]，长度=[5, 3, 7, 6, 3, 5]，最大绝对权重=2.860


## 手写核心实现与中间量

核心实现保留 state、ratio、优势、mask 或概率分母等中间量，不用 Trainer 或现成 Agent/Attention 框架遮蔽机制。


In [3]:
normalized_terms = [sum(min(ratio * advantage, max(0.8, min(1.2, ratio)) * advantage) for ratio in ratios) / len(ratios) for ratios, advantage in zip(token_ratios, advantages)]  # 对每条完整回复按 token 数求平均。
group_objective = sum(normalized_terms) / len(normalized_terms)  # 再对所有 sequence 平均得到组训练目标。
core_metric = max(abs(value) for value in normalized_terms)  # 记录长度归一后的最大样本权重。
print(f'GRPO sequence-normalized terms={ [round(value, 3) for value in normalized_terms] }，组目标={group_objective:.4f}')  # 输出长度公平后的核心结果。


GRPO sequence-normalized terms=[0.324, -0.38, 0.112, 0.477, -0.51, 0.14]，组目标=0.0271


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 建立基线与核心的同口径结果表。
for name, metric in comparison_rows:  # 逐行输出结果表。
    print(f'{name:<8} | 指标={metric:.6f}')  # 显示可读数值对照。


Baseline | 指标=2.860000
核心机制     | 指标=0.510000


## 结果解读

这里只能得出本受控样本上的机制结论。生产训练需记录每个样本 token 数、优势、ratio、clip fraction 与按长度分桶的 loss；只报全局平均会隐藏长度偏置。 生产决策必须进一步看验证集、线上安全指标、算力和版本可追溯性。

## 失败案例

下方先让关键条件真实失效，再展示修复如何改变可观测指标。


In [5]:
long_index = 2  # 选择长度为七的同 prompt 长回答。
failure_metric = abs(token_sum_terms[long_index])  # 读取它在 token-sum 下的累计权重。
fix_metric = abs(normalized_terms[long_index])  # 读取它按 sequence 平均后的权重。
print(f'失败：长度={token_lengths[long_index]} 的 token-sum 权重={failure_metric:.3f}；修复：sequence-normalized={fix_metric:.3f}')  # 展示同一回答仅因长度不同被放大。


失败：长度=7 的 token-sum 权重=0.784；修复：sequence-normalized=0.112


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产训练需记录每个样本 token 数、优势、ratio、clip fraction 与按长度分桶的 loss；只报全局平均会隐藏长度偏置。

**常见坑：** 把 reward 相同的 token advantage 当作 token-level reward，或把长度归一化放在不同层级导致不同实现不可比较。

**延伸追问：** 对 reasoning token 和 final answer token 应否使用相同权重？长度奖励/惩罚如何与 GRPO 的归一化互动？

## 生产差距

实验运行于 CPU/FP32，只有 6 条离线样本，省略了真实 rollout、分布式同步、混合精度、内容安全、数据治理、checkpoint 和监控。上线版本应以受审计的状态、指标和回滚流程替代这些教学变量。


In [6]:
assert len(normalized_terms) == 6  # 验证六条回复都得到 GRPO surrogate。
assert core_metric < baseline_metric  # 验证长度归一缩小了样本权重极值。
assert failure_metric > fix_metric  # 验证长回复 token-sum 被过度加权。
assert len(advantages) == 6  # 验证优势在各 prompt 组内计算。
